In [1]:
# ============================================================
# BattingEdge V9.5 - TRANSFORMER Model Training
# Features: 99 raw pose + 8 angles (NO velocities)
# Target: 85-90% accuracy (State-of-the-Art)
# ============================================================

# ✅ FIX: Import Pandas first to prevent circular import errors
import pandas as pd 
import numpy as np
import pickle
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

# ================= CONFIG =================
FEATURE_DIR = Path(r"D:\Users\Anoshia\BattingEdge_FYP\features")
MODEL_DIR   = Path(r"D:\Users\Anoshia\BattingEdge_FYP\backend\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Algorithm Name
ALGO_NAME = "transformer"

# Hyperparameters
EPOCHS = 70
BATCH_SIZE = 16
LEARNING_RATE = 1e-4  # Transformers usually need lower LR
NUM_HEADS = 4         # Attention heads
FF_DIM = 128          # Feed forward dimension
DROPOUT = 0.3
# ==========================================

print("="*70)
print(f"BATTINGEDGE V9.5 - MODEL TRAINING ({ALGO_NAME.upper()})")
print("="*70)
print()

# ================= LOAD DATA =================
print("📂 Loading data...")

try:
    X_train = np.load(FEATURE_DIR / "X_train.npy")
    y_train = np.load(FEATURE_DIR / "y_train.npy")
    X_val   = np.load(FEATURE_DIR / "X_val.npy")
    y_val   = np.load(FEATURE_DIR / "y_val.npy")
    X_test  = np.load(FEATURE_DIR / "X_test.npy")
    y_test  = np.load(FEATURE_DIR / "y_test.npy")

    with open(FEATURE_DIR / "classes.pkl", "rb") as f:
        CLASSES = pickle.load(f)
except FileNotFoundError as e:
    print(f"\n❌ CRITICAL ERROR: Could not find data files.")
    print(f"   Checked directory: {FEATURE_DIR}")
    raise e

num_classes = len(CLASSES)
T, F = X_train.shape[1], X_train.shape[2]

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {CLASSES}")
print(f"Features per frame: {F}")
print()

# ================= CRITICAL: SCALING =================
print("⚖️  Applying StandardScaler...")

scaler = StandardScaler()

# Fit on training data (flatten to 2D)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train * T, F)
scaler.fit(X_train_2d)

def scale_data(X):
    N, T, F = X.shape
    X_2d = X.reshape(N * T, F)
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(N, T, F)

X_train = scale_data(X_train)
X_val   = scale_data(X_val)
X_test  = scale_data(X_test)

# Save scaler
scaler_path = MODEL_DIR / f"scaler_V9_5_{ALGO_NAME}.pkl"
classes_path = MODEL_DIR / f"classes_V9_5_{ALGO_NAME}.pkl"
joblib.dump(scaler, scaler_path)
joblib.dump(CLASSES, classes_path)
print(f"✅ Scaler saved: {scaler_path.name}")
print()

# ================= CLASS WEIGHTS =================
print("⚖️  Computing class weights...")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("   Weights loaded.")
print()

# ================= MODEL: TRANSFORMER =================
print("🏗️  Building Transformer model...")

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

def build_transformer(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    
    # Transformer Block 1
    x = transformer_encoder(inputs, head_size=128, num_heads=NUM_HEADS, ff_dim=FF_DIM, dropout=DROPOUT)
    
    # Transformer Block 2 (Stacked for depth)
    x = transformer_encoder(x, head_size=128, num_heads=NUM_HEADS, ff_dim=FF_DIM, dropout=DROPOUT)
    
    # Global Average Pooling (Flattening sequence)
    x = layers.GlobalAveragePooling1D()(x)
    
    # Classification Head
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    
    return models.Model(inputs, outputs)

model = build_transformer((T, F), num_classes)

model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print()

# ================= CALLBACKS =================
print("⚙️  Setting up callbacks...")

checkpoint_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_best.keras"

callbacks_list = [
    callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(str(checkpoint_path), monitor="val_accuracy", save_best_only=True, verbose=1)
]

print(f"✅ Checkpoint: {checkpoint_path.name}")
print()

# ================= TRAINING =================
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)
print()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    verbose=1
)

print()
print("="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print()

# Save final model
final_path = MODEL_DIR / f"battingedge_V9_5_{ALGO_NAME}_final.keras"
model.save(final_path)
print(f"💾 Saved final model: {final_path.name}")
print()

# ================= EVALUATION =================
print("="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70)
print()

# Load best model
best_model = tf.keras.models.load_model(str(checkpoint_path))
print(f"✅ Loaded best model from: {checkpoint_path.name}")
print()

# Predict
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

# Classification report
print("📋 CLASSIFICATION REPORT:")
print()
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

report_path = MODEL_DIR / f"report_V9_5_{ALGO_NAME}.txt"
with open(report_path, "w") as f:
    f.write(report)
print(f"✅ Report saved: {report_path.name}")
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("🔢 CONFUSION MATRIX:")
print("   (Rows = True, Cols = Predicted)")
print()
print("        ", "  ".join([f"{cls[:4]:>4s}" for cls in CLASSES]))
for i, cls in enumerate(CLASSES):
    print(f"{cls[:8]:8s}", "  ".join([f"{cm[i,j]:4d}" for j in range(num_classes)]))
print()

# Per-class accuracy
print("📈 PER-CLASS ACCURACY:")
print()
for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"   {cls:15s}: {correct:3d}/{total:3d} = {accuracy:5.1f}%")

overall_acc = np.trace(cm) / np.sum(cm) * 100
print()
print(f"   {'OVERALL':15s}: {np.trace(cm):3d}/{np.sum(cm):3d} = {overall_acc:5.2f}%")
print()

# Major confusions
print("🔍 MAJOR CONFUSIONS (>3 cases):")
print()
confusions = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 3:
            confusions.append((CLASSES[i], CLASSES[j], cm[i, j]))

if confusions:
    confusions.sort(key=lambda x: x[2], reverse=True)
    for true_cls, pred_cls, count in confusions:
        print(f"   {true_cls:15s} → {pred_cls:15s}: {count} cases")
else:
    print("   ✅ No major confusions!")
print()

# ================= VISUALIZATIONS =================
print("📊 Generating visualizations...")

# Confusion matrix heatmap
cm_path = MODEL_DIR / f"confusion_matrix_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f"Confusion Matrix - V9.5 ({ALGO_NAME})", fontsize=14, fontweight='bold')
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(cm_path, dpi=300)
print(f"   ✅ Saved: {cm_path.name}")
plt.close()

# Training history
history_path = MODEL_DIR / f"training_history_V9_5_{ALGO_NAME}.png"
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(history_path, dpi=300)
print(f"   ✅ Saved: {history_path.name}")
plt.close()

print()

# ================= METADATA =================
print("="*70)
print(f"🎉 V9.5 ({ALGO_NAME}) TRAINING COMPLETE")
print("="*70)
print()

# Save metadata
metadata = {
    "version": "V9.5",
    "algorithm": ALGO_NAME,
    "features": "99 raw pose + 8 angles (no velocities)",
    "classes": CLASSES,
    "test_accuracy": float(overall_acc),
    "per_class_accuracy": {
        CLASSES[i]: float((cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0)
        for i in range(num_classes)
    },
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "heads": NUM_HEADS
    }
}

metadata_path = MODEL_DIR / f"metadata_V9_5_{ALGO_NAME}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("📦 Saved artifacts:")
print(f"   1. Best model: {checkpoint_path.name}")
print(f"   2. Final model: {final_path.name}")
print(f"   3. Metadata: {metadata_path.name}")
print()

if overall_acc >= 85:
    print("✨ EXCELLENT RESULT! Transformer power unlocked.")
elif overall_acc >= 81:
    print("✅ GOOD RESULT. Competitive with LSTM.")
else:
    print("⚠️ Below target. Transformers need lots of data.")

print("="*70)

BATTINGEDGE V9.5 - MODEL TRAINING (TRANSFORMER)

📂 Loading data...
Train: 3007 samples
Val:   388 samples
Test:  378 samples
Classes: ['Cover Drive', 'Cut Shot', 'Defense', 'Pull Shot', 'Sweep Shot']
Features per frame: 107

⚖️  Applying StandardScaler...
✅ Scaler saved: scaler_V9_5_transformer.pkl

⚖️  Computing class weights...
   Weights loaded.

🏗️  Building Transformer model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50, 107)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 107)   │    220,779 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 50, 107)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 107)   │        214 │ dropout_1[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 107)   │          0 │ layer_normalizat… │
│                     │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 128)   │     13,824 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 50, 128)   │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 107)   │     13,803 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 107)   │        214 │ conv1d_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 107)   │          0 │ layer_normalizat… │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 107)   │    220,779 │ add_1[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 50, 107)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 107)   │        214 │ dropout_4[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 107)   │          0 │ layer_normalizat… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 50, 128)   │     13,824 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 50, 128)   │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 50, 107)   │     13,803 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 107)   │        214 │ conv1d_3[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 107)   │          0 │ layer_normalizat… │
│                     │                   │            │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 505,161 (1.93 MB)

 Trainable params: 505,033 (1.93 MB)

 Non-trainable params: 128 (512.00 B)


⚙️  Setting up callbacks...
✅ Checkpoint: battingedge_V9_5_transformer_best.keras

🚀 STARTING TRAINING

Epoch 1/70
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.2868 - loss: 1.9858
Epoch 1: val_accuracy improved from None to 0.46392, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_transformer_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 55s 132ms/step - accuracy: 0.3322 - loss: 1.8007 - val_accuracy: 0.4639 - val_loss: 1.3738 - learning_rate: 1.0000e-04
Epoch 2/70
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.4267 - loss: 1.5346
Epoch 2: val_accuracy improved from 0.46392 to 0.52577, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_transformer_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 23s 123ms/step - accuracy: 0.4333 - loss: 1.5307 - val_accuracy: 0.5258 - val_loss: 1.3236 - learning_rate: 1.0000e-04
Epoch 3/70
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.4749 - loss: 1.3955
Epoch 3: val_accur